# Pipeline VLM Mamografía — Evaluación Zero-Shot
**Autor:** Luis Enrique Medrano Santana  
**Asesor:** Luis Vives Garnique  
**Institución:** Pontificia Universidad Católica del Perú (PUCP)  
**Título:** Modelo generativo multimodal de visión-lenguaje orientado al apoyo a la toma de decisiones clínicas en el diagnóstico de cáncer de mama

---

## Descripción

Este notebook implementa el pipeline completo de **evaluación zero-shot** sobre el dataset VinDr-Mammo.
Comprende desde la definición de los dataframes hasta la evaluación comparativa de tres
modelos de visión-lenguaje (VLM) en modalidad zero-shot, sin ningún fine-tuning previo.

## Estructura

| Sección | Descripción |
|---------|-------------|
| 0 | Instalación de dependencias |
| 1 | Imports globales y configuración de rutas |
| 2 | Validación de integridad del dataset |
| 3 | Evaluación zero-shot: MedGemma 4B, Qwen2.5-VL 7B, HuatuoGPT-Vision 7B |

## Formato del merge 2×2

```
┌──────────────┬──────────────┐
│  R_CC        │  L_CC        │
│ (mama dcha,  │ (mama izq,   │
│  vista CC)   │  vista CC)   │
├──────────────┼──────────────┤
│  R_MLO       │  L_MLO       │
│ (mama dcha,  │ (mama izq,   │
│  vista MLO)  │  vista MLO)  │
└──────────────┴──────────────┘
```

---
## Sección 0 — Instalación de dependencias

Instala todas las bibliotecas necesarias con versiones fijadas para garantizar reproducibilidad.

**Reiniciar el runtime después de ejecutar esta celda.**

In [ ]:
# ── Instalación de dependencias con versiones fijadas ─────────────────────────

!pip install -q \
    "transformers>=5.4.0" \
    "accelerate>=0.26.0" \
    "bitsandbytes>=0.43.0" \
    "peft>=0.10.0" \
    "qwen-vl-utils>=0.0.8" \
    "bert-score" \
    "rouge-score" \
    "scikit-learn" \
    "scipy" \
    "pydicom" \
    "opencv-python-headless" \
    "Pillow>=10.0" \
    "tqdm" \
    "sentencepiece" \
    "protobuf"

print('Instalación completa. Reinicia el runtime antes de continuar.')

---
## Sección 1 — Imports globales y configuración

Centraliza todos los imports y variables de configuración.
Las secciones siguientes asumen que esta celda ya fue ejecutada.

**Rutas principales en Google Drive:**
- `images/` — DICOMs originales organizados por `study_id/`
- `processed_dcm/` — PNGs merge 2×2 tras preprocesamiento
- `metadata/` — CSVs de anotaciones VinDr-Mammo
- `results/` — JSONs con resultados de evaluación

In [ ]:
# ── Montar Google Drive ───────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Imports ──────────────────────────────────────────────────────────
import os
import gc
import re
import json
import time
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

import cv2
import pydicom
from PIL import Image
import torch
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────────────
# RUTAS — Modificable
# ─────────────────────────────────────────────────────────────────────────────
DRIVE_ROOT = '/content/drive/MyDrive/vindr_1000'
IMAGES_ROOT = f'{DRIVE_ROOT}/images'
PROCESSED_DIR = f'{DRIVE_ROOT}/processed_dcm'
META_DIR = f'{DRIVE_ROOT}/metadata'
RESULTS_DIR = f'{DRIVE_ROOT}/results'

# CSVs de VinDr-Mammo
META_CSV = f'{META_DIR}/selected_1000_studies.csv'
BREAST_CSV = f'{META_DIR}/breast-level_annotations.csv'
FINDING_CSV = f'{META_DIR}/finding_annotations.csv'
REPORTS_CSV = f'{META_DIR}/synthetic_reports_1000.csv'
VAL_CSV = f'{META_DIR}/validation_report.csv'

# Crear carpetas de salida si no existen
for d in [PROCESSED_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# CREDENCIALES
# HF_TOKEN: token de HuggingFace con acceso aprobado a google/medgemma-4b-it
# ─────────────────────────────────────────────────────────────────────────────
HF_TOKEN = '...'

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURACIÓN GLOBAL
# ─────────────────────────────────────────────────────────────────────────────
RANDOM_SEED = 20201531
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# ── Verificación del entorno ──────────────────────────────────────────────────
print('Configuración global cargada')
print(f'   PyTorch : {torch.__version__}')
print(f'   CUDA : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cargar metadatos y crear DataFrames globales ──────────────────────────────
# Estos DataFrames se reutilizan en todas las secciones del pipeline.
# breast_df contiene una fila por vista (4 filas por estudio).
# meta_df contiene una fila por estudio con el BIRADS y densidad consolidados.

meta_df = pd.read_csv(META_CSV)
breast_df = pd.read_csv(BREAST_CSV)

# Normalizar nombre de columna BIRADS
if 'birads_max' in meta_df.columns:
    meta_df = meta_df.rename(columns={'birads_max': 'birads'})

# Normalizar a valores escalares: 'BI-RADS 2' → 2, 'DENSITY C' → 'C'
meta_df['birads_norm'] = meta_df['birads'].str.extract(r'(\d)').astype(int)
meta_df['density_norm'] = meta_df['density'].str.strip().str[-1]

# Separar splits
train_df = meta_df[meta_df['our_split'] == 'train'].reset_index(drop=True)
test_df = meta_df[meta_df['our_split'] == 'test'].reset_index(drop=True)

print(f'Total estudios: {len(meta_df)}')
print(f'Train: {len(train_df)}')
print(f'Test: {len(test_df)}')
print(f'\nDistribución BI-RADS:')
print(meta_df['birads_norm'].value_counts().sort_index())
print(f'\nDistribución densidad ACR:')
print(meta_df['density_norm'].value_counts().sort_index())

---
## Sección 2 — Validación de integridad del dataset

Verifica que los estudios cumplen 5 condiciones antes de procesar:

1. El directorio del estudio existe en Drive
2. Los DICOMs son legibles (headers válidos)
3. Las 4 vistas requeridas están presentes: R_CC, L_CC, R_MLO, L_MLO
4. Las etiquetas BI-RADS (1-5) y densidad (A-D) son válidas
5. El PNG del merge ya existe y no está corrupto

**Output:** `validation_report.csv` con el estado de cada estudio.  
**Resiliencia:** si `validation_report.csv` ya existe, se omite la validación (`skip_if_exists=True`).

In [ ]:
# ── Validación de integridad de los estudios ──────────────────────────────────
# Si validation_report.csv ya existe, se carga directamente.

SKIP_VALIDATION = Path(VAL_CSV).exists()

if SKIP_VALIDATION:
    print('Validación ya completada — cargando reporte existente')
    val_df = pd.read_csv(VAL_CSV)
else:
    # Vistas requeridas para construir el merge 2×2
    VALID_BIRADS = {1, 2, 3, 4, 5}
    VALID_DENSITY = {'A', 'B', 'C', 'D'}
    REQUIRED_VIEWS = {('L', 'CC'), ('R', 'CC'), ('L', 'MLO'), ('R', 'MLO')}

    results = []
    for _, row in tqdm(meta_df.iterrows(), total=len(meta_df), desc='Validando'):
        study_id = str(row['study_id'])
        study_dir = Path(IMAGES_ROOT) / study_id
        issues = []

        record = {
            'study_id': study_id, 'dir_exists': False,
            'n_readable': 0, 'n_corrupt': 0,
            'has_4_views': False, 'missing_views': '',
            'has_valid_birads': False, 'has_valid_density': False,
            'png_exists': False, 'png_valid': False,
            'status': 'UNKNOWN', 'issues': ''
        }

        # CHECK 1: Directorio del estudio existe
        if not study_dir.exists():
            record.update({'issues': 'DIR_MISSING', 'status': 'FAIL'})
            results.append(record)
            continue
        record['dir_exists'] = True

        # CHECK 2: DICOMs legibles — solo se leen headers para eficiencia
        dicom_files = list(study_dir.glob('*.dicom')) or list(study_dir.glob('*.dcm'))
        readable = corrupt = 0
        for dcm in dicom_files:
            try:
                ds = pydicom.dcmread(str(dcm), stop_before_pixels=True)
                readable += 1 if hasattr(ds, 'Rows') else 0
                corrupt += 0 if hasattr(ds, 'Rows') else 1
            except Exception:
                corrupt += 1
        record.update({'n_readable': readable, 'n_corrupt': corrupt})
        if readable == 0:
            issues.append('ALL_CORRUPT')

        # CHECK 3: Las 4 vistas presentes (R_CC, L_CC, R_MLO, L_MLO)
        study_rows    = breast_df[breast_df['study_id'] == study_id]
        present_views = set()
        for _, brow in study_rows.iterrows():
            lat = str(brow.get('laterality',    '')).strip().upper()
            view = str(brow.get('view_position', '')).strip().upper()
            img_id = str(brow.get('image_id', ''))
            dcm_path = study_dir / f'{img_id}.dicom'
            if not dcm_path.exists():
                dcm_path = study_dir / f'{img_id}.dcm'
            if lat and view and dcm_path.exists():
                present_views.add((lat, view))
        missing = REQUIRED_VIEWS - present_views
        record.update({'has_4_views': len(missing) == 0,
                       'missing_views': '|'.join(f'{l}_{v}' for l, v in missing)})
        if missing:
            issues.append(f'MISSING_VIEWS({len(missing)})')

        # CHECK 4: Etiquetas BIRADS y densidad válidas
        birads_match = re.search(r'\b([1-5])\b', str(row.get('birads', '')))
        density_match = re.search(r'\b([ABCD])\b', str(row.get('density', '')).upper())
        record['has_valid_birads']  = birads_match is not None
        record['has_valid_density'] = density_match is not None
        if not record['has_valid_birads']:  issues.append('INVALID_BIRADS')
        if not record['has_valid_density']: issues.append('INVALID_DENSITY')

        # CHECK 5: PNG merge 2×2 ya existe y es válido
        png_path = Path(PROCESSED_DIR) / f'{study_id}.png'
        record['png_exists'] = png_path.exists()
        if png_path.exists():
            try:
                img = Image.open(str(png_path))
                img.verify()
                record['png_valid'] = True
            except Exception:
                issues.append('PNG_CORRUPT')

        # Estado final: OK / WARN (usable con limitaciones) / FAIL
        record['issues'] = '|'.join(issues) if issues else 'OK'
        if not issues:
            record['status'] = 'OK'
        elif 'ALL_CORRUPT' in record['issues'] or 'DIR_MISSING' in record['issues']:
            record['status'] = 'FAIL'
        else:
            record['status'] = 'WARN' if (
                len(present_views) >= 2 and
                record['has_valid_birads'] and
                record['has_valid_density']
            ) else 'FAIL'

        results.append(record)

    val_df = pd.DataFrame(results)
    val_df.to_csv(VAL_CSV, index=False)
    print(f'Reporte guardado: {VAL_CSV}')

# Resumen de validación
counts = val_df['status'].value_counts()
print(f'\nResumen de validación:')
print(f'  OK      : {counts.get("OK",   0)}')
print(f'  WARN    : {counts.get("WARN", 0)}')
print(f'  FAIL    : {counts.get("FAIL", 0)}')
print(f'  Usables : {counts.get("OK",0) + counts.get("WARN",0)}')

---
## Sección 3 — Evaluación Zero-Shot: 3 modelos VLM

Evalúa tres modelos de visión-lenguaje en modalidad **zero-shot**, sin fine-tuning previo,
sobre los estudios de VinDr-Mammo procesados.

### Modelos evaluados

| Modelo | Arquitectura | Preentrenamiento |
|--------|-------------|------------------|
| **MedGemma 4B-it** | MedSigLIP 400M + Gemma 3 4B | 33M pares médicos imagen-texto |
| **Qwen2.5-VL 7B** | ViT dinámico + Qwen2.5 7B | General (~4.1T tokens multimodales) |
| **HuatuoGPT-Vision 7B** | Igual a Qwen2.5-VL | PubMedVision 1.3M pares imagen-texto |


### Métricas

| Métrica | Descripción |
|---------|-------------|
| **BERTScore F1** | Coherencia semántica clínica (SciBERT, num_layers=9) |
| **BERTScore CI 95%** | Intervalo de confianza bootstrap (1000 muestras) |
| **ROUGE-L** | Similitud léxica de secuencias más largas comunes |
| **BI-RADS Accuracy** | Exactitud global de clasificación BI-RADS |
| **BI-RADS Macro F1** | F1 macro penalizando clases minoritarias |
| **Density Accuracy** | Exactitud global de clasificación de densidad ACR |
| **Density Macro F1** | F1 macro penalizando clases de densidad minoritarias |

### Prompt unificado

El mismo prompt se usa para los 3 modelos para garantizar comparabilidad.
El layout del merge coincide exactamente con el producido:

```
top-left = RIGHT breast, CC view
top-right = LEFT breast, CC view
bot-left = RIGHT breast, MLO view
bot-right = LEFT breast, MLO view
```

In [ ]:
# ── Parche BERTScore para Python 3.12 ────────────────────────────────────────
# PROBLEMA: allenai/scibert_scivocab_uncased tiene model_max_length=1e30
# SOLUCIÓN: interceptar sent_encode y forzar max_length=512.

import bert_score.utils as bs_utils

def _patched_sent_encode(tokenizer, sent):
    sent = sent.strip()
    if sent == '':
        return tokenizer.encode('', add_special_tokens=True)
    return tokenizer.encode(
        sent,
        add_special_tokens=True,
        max_length=512,
        truncation=True,
    )

bs_utils.sent_encode = _patched_sent_encode
print('Parche BERTScore aplicado (max_length=512).')

In [ ]:
# ── Prompts del sistema para evaluación zero-shot ─────────────────────────────
# El mismo prompt se usa para los 3 modelos para garantizar comparabilidad.
# El layout del merge es idéntico al producido en la Sección 3:
#   top-left = RIGHT CC | top-right = LEFT CC
#   bot-left = RIGHT MLO | bot-right = LEFT MLO

SYSTEM_PROMPT = (
    "You are a board-certified breast radiologist with extensive experience "
    "in screening mammography. "
    "The image shows four mammographic views arranged in a 2×2 grid:\n"
    "  top-left = RIGHT breast, cranio-caudal (CC) view\n"
    "  top-right = LEFT breast, cranio-caudal (CC) view\n"
    "  bottom-left = RIGHT breast, medio-lateral oblique (MLO) view\n"
    "  bottom-right = LEFT breast, medio-lateral oblique (MLO) view\n"
    "Analyze all four views together before generating the report. "
    "Respond ONLY with a valid JSON object. No text outside the JSON."
)

USER_PROMPT = (
    "Generate a structured mammography report analyzing all four views.\n"
    "Return ONLY this JSON - no markdown, no explanation:\n"
    '{\n'
    '  \"breast_density\": \"<A|B|C|D>\",\n'
    '  \"findings\": \"<narrative findings in 2-3 sentences>\",\n'
    '  \"birads\": \"<1|2|3|4|5>\",\n'
    '  \"suspicion\": \"<low|intermediate|high>\"\n'
    '}'
)

print('Prompts definidos.')
print('\nSYSTEM_PROMPT:')
print(SYSTEM_PROMPT)
print('\nUSER_PROMPT:')
print(USER_PROMPT)

In [ ]:
# ── Utilidades compartidas de evaluación ──────────────────────────────────────

def load_image(study_id: str) -> Image.Image | None:
    """
    Carga el PNG merge 2×2 procesado de un estudio desde Drive.
    """
    p = Path(PROCESSED_DIR) / f'{study_id}.png'
    return Image.open(p).convert('RGB') if p.exists() else None


def parse_response(text: str) -> dict:
    """
    Extrae el JSON de la respuesta del modelo de forma tolerante.

    Intenta 3 estrategias en orden:
    1. JSON puro directo (caso ideal)
    2. Bloque ```json ... ``` (cuando el modelo añade markdown)
    3. Primer { ... } encontrado en el texto (último recurso)
    """
    if not text:
        return {}
    # Estrategia 1: JSON puro
    try:
        return json.loads(text.strip())
    except Exception:
        pass
    # Estrategia 2: bloque ```json ... ```
    m = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass
    # Estrategia 3: primer { ... } en el texto
    m = re.search(r'\{[^{}]*\}', text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            pass
    return {}


def norm_birads(raw) -> str | None:
    """Normaliza BI-RADS a dígito 1-5. Ej: 'BI-RADS 4A' → '4'."""
    if raw is None:
        return None
    m = re.search(r'([1-5])', str(raw).upper())
    return m.group(1) if m else None


def norm_density(raw) -> str | None:
    """Normaliza densidad ACR a letra mayúscula A-D. Ej: 'DENSITY C' → 'C'."""
    if raw is None:
        return None
    m = re.search(r'\b([ABCD])\b', str(raw).upper())
    return m.group(1) if m else None


def compute_metrics(records: list) -> dict:
    """
    Calcula métricas de evaluación sobre una lista de registros.
    """
    from bert_score import score as bert_fn
    from rouge_score import rouge_scorer as rouge_lib

    preds = [r.get('pred_report', '') or '' for r in records]
    refs = [r.get('gt_report', '') or '' for r in records]

    if not records or not any(p.strip() for p in preds):
        print('Sin predicciones válidas — retornando métricas en cero')
        return {k: 0.0 for k in [
            'bertscore_f1', 'bertscore_ci_low', 'bertscore_ci_high',
            'rouge_l', 'birads_acc', 'birads_macro_f1',
            'density_acc', 'density_macro_f1',
            'n_evaluated', 'n_birads_valid', 'n_density_valid'
        ]}

    # ── BERTScore con SciBERT (coherencia semántica clínica) ──────────────────
    print(' Calculando BERTScore (SciBERT, num_layers=9)...')
    _, _, F1 = bert_fn(
        preds, refs,
        model_type='allenai/scibert_scivocab_uncased',
        num_layers=9,
        lang='en',
        verbose=False,
        device=DEVICE,
    )
    bs_f1 = float(F1.mean())
    f1_vals = F1.numpy()

    # Bootstrap CI 95% para BERTScore F1 (1000 muestras)
    boot_means = [
        np.mean(np.random.choice(f1_vals, len(f1_vals), replace=True))
        for _ in range(1000)
    ]
    ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])

    # ── ROUGE-L (similitud léxica) ────────────────────────────────────────────
    print(' Calculando ROUGE-L...')
    scorer = rouge_lib.RougeScorer(['rougeL'], use_stemmer=True)
    rls = [
        scorer.score(r, p)['rougeL'].fmeasure if p.strip() and r.strip() else 0.0
        for p, r in zip(preds, refs)
    ]
    rl = float(np.mean(rls))

    # ── BI-RADS accuracy y macro F1 ──────────────────────────────────────────
    birads_pairs = [
        (r['pred_birads'], r['gt_birads']) for r in records
        if r.get('pred_birads') and r.get('gt_birads')
    ]
    if birads_pairs:
        pb, gb = zip(*birads_pairs)
        birads_acc = accuracy_score(gb, pb)
        birads_f1 = f1_score(list(gb), list(pb), average='macro', zero_division=0)
    else:
        birads_acc = birads_f1 = 0.0

    # ── Density accuracy y macro F1 ──────────────────────────────────────────
    density_pairs = [
        (r['pred_density'], r['gt_density']) for r in records
        if r.get('pred_density') and r.get('gt_density')
    ]
    if density_pairs:
        pd_, gd = zip(*density_pairs)
        density_acc = accuracy_score(gd, pd_)
        density_f1 = f1_score(list(gd), list(pd_), average='macro', zero_division=0)
    else:
        density_acc = density_f1 = 0.0

    return {
        'bertscore_f1': round(bs_f1, 4),
        'bertscore_ci_low': round(ci_low, 4),
        'bertscore_ci_high': round(ci_high, 4),
        'rouge_l': round(rl, 4),
        'birads_acc': round(birads_acc, 4),
        'birads_macro_f1': round(birads_f1, 4),
        'density_acc': round(density_acc, 4),
        'density_macro_f1': round(density_f1, 4),
        'n_evaluated': len(records),
        'n_birads_valid': len(birads_pairs),
        'n_density_valid': len(density_pairs),
    }


def run_zeroshot(model_name: str, generate_fn, output_key: str,
                  eval_df: pd.DataFrame, skip_if_exists: bool = True) -> dict:
    """
    Bucle de evaluación zero-shot reutilizable para cualquier modelo VLM.

    Protocolo:
    1. Iterar sobre eval_df cargando el PNG de cada estudio
    2. Generar reporte con generate_fn
    3. Parsear respuesta y extraer BI-RADS y densidad
    4. Calcular métricas al final del bucle
    5. Guardar JSON completo en RESULTS_DIR/{output_key}.json
    """
    out_path = f'{RESULTS_DIR}/{output_key}.json'

    if skip_if_exists and Path(out_path).exists():
        print(f'{model_name} — resultados existentes cargados')
        with open(out_path) as f:
            return json.load(f)['metrics']

    print(f'\n{"═"*60}')
    print(f'  {model_name} — evaluando {len(eval_df)} estudios')
    print(f'{"═"*60}')

    raw_results = []
    eval_records = []
    skipped = 0
    t0 = time.time()

    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc=model_name):
        study_id = str(row['study_id'])
        gt_birads = str(row['birads_norm'])
        gt_density = str(row['density_norm'])

        img = load_image(study_id)
        if img is None:
            skipped += 1
            continue

        try:
            raw_text = generate_fn(img)
        except Exception as e:
            print(f' {study_id}: {e}')
            skipped += 1
            continue

        parsed = parse_response(raw_text)
        pred_birads = norm_birads(parsed.get('birads'))
        pred_density = norm_density(parsed.get('breast_density'))
        pred_report = parsed.get('findings', raw_text[:400] if raw_text else '')

        # Recuperar reporte sintético de referencia para BERTScore y ROUGE-L
        gt_row = reports_df[reports_df['study_id'] == study_id]
        gt_report = gt_row['full_report'].iloc[0] if not gt_row.empty else ''

        raw_results.append({
            'study_id': study_id,
            'raw_output': raw_text,
            'parsed': parsed,
        })
        eval_records.append({
            'study_id': study_id,
            'pred_report': pred_report,
            'gt_report': gt_report,
            'pred_birads': pred_birads,
            'gt_birads': gt_birads,
            'pred_density': pred_density,
            'gt_density': gt_density,
        })

    elapsed = (time.time() - t0) / 60
    print(f'\n Procesados: {len(eval_records)} | Saltados: {skipped} | {elapsed:.1f} min')

    print(' Calculando métricas...')
    metrics = compute_metrics(eval_records)

    print(f'\n Resultados {model_name}:')
    for k, v in metrics.items():
        print(f'  {k:<25}: {v}')

    # Guardar JSON completo en Drive — incluye outputs crudos para auditoría
    output = {
        'model': model_name,
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
        'n_evaluated': len(eval_df),
        'n_skipped': skipped,
        'metrics': metrics,
        'raw_results': raw_results,
    }
    with open(out_path, 'w') as f:
        json.dump(output, f, indent=2, ensure_ascii=False)
    print(f'\n  Guardado: {out_path}')
    return metrics


print('Utilidades de evaluación zero-shot definidas.')

In [ ]:
# ── Modelo 1: MedGemma 4B-it ──────────────────────────────────────────────────
# Arquitectura : MedSigLIP 400M (encoder visual) + Gemma 3 4B (LLM)
# Acceso       : restringido en HuggingFace — requiere aprobación previa

import gc
from transformers import AutoProcessor, AutoModelForImageTextToText

MEDGEMMA_ID = 'google/medgemma-4b-it'


def generate_medgemma(img: Image.Image) -> str:
    """
    Genera un reporte mammográfico con MedGemma 4B-it.
    Usa el chat template nativo del modelo con generación determinista (do_sample=False).
    """
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'text',  'text': SYSTEM_PROMPT},
            {'type': 'image', 'image': img},
            {'type': 'text',  'text': USER_PROMPT},
        ]
    }]
    inputs = mg_processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors='pt',
        return_dict=True,
    ).to(DEVICE)

    with torch.inference_mode():
        out = mg_model.generate(**inputs, max_new_tokens=300, do_sample=False)

    text = mg_processor.decode(out[0], skip_special_tokens=True)
    # Extraer solo la parte generada (eliminar el prompt de entrada)
    return text.split('model')[-1].strip()


# Cargar MedGemma con token de HuggingFace
print('Cargando MedGemma 4B-it...')
mg_processor = AutoProcessor.from_pretrained(MEDGEMMA_ID, token=HF_TOKEN)
mg_model = AutoModelForImageTextToText.from_pretrained(
    MEDGEMMA_ID, token=HF_TOKEN,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
mg_model.eval()
print(f'MedGemma cargado — VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# Cargar reportes sintéticos si no están en memoria
if 'reports_df' not in dir():
    reports_df = pd.read_csv(REPORTS_CSV)

# Evaluar sobre todos los estudios
metrics_mg = run_zeroshot(
    model_name = 'MedGemma-4B-it',
    generate_fn = generate_medgemma,
    output_key = 'o1_medgemma_4b',
    eval_df = meta_df,
    skip_if_exists= True,
)

# Liberar VRAM antes del siguiente modelo
del mg_model, mg_processor
gc.collect()
torch.cuda.empty_cache()
vram_free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
print(f'VRAM disponible: {vram_free:.1f} GB')

In [ ]:
# ── Modelo 2: Qwen2.5-VL 7B-Instruct ─────────────────────────────────────────
# Arquitectura : ViT dinámico nativo + Qwen2.5 7B
# Preentrenamiento: propósito general (~4.1T tokens multimodales)

from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

QWEN_ID = 'Qwen/Qwen2.5-VL-7B-Instruct'

def generate_qwen(img: Image.Image) -> str:
    """
    Genera un reporte mammográfico con Qwen2.5-VL 7B.
    Usa process_vision_info para el manejo correcto de imágenes con ViT dinámico.
    """
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'text',  'text': SYSTEM_PROMPT},
            {'type': 'image', 'image': img},
            {'type': 'text',  'text': USER_PROMPT},
        ]
    }]
    text_in = qwen_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, _ = process_vision_info(messages)
    inputs = qwen_processor(
        text=[text_in], images=image_inputs,
        return_tensors='pt', padding=True,
    ).to(DEVICE)

    with torch.inference_mode():
        out = qwen_model.generate(**inputs, max_new_tokens=300, do_sample=False)

    # Decodificar solo los tokens nuevos (excluir el prompt de entrada)
    trimmed = out[0][inputs.input_ids.shape[1]:]
    return qwen_processor.decode(trimmed, skip_special_tokens=True).strip()


# Cargar Qwen2.5-VL (sin token, es de acceso público)
print('Cargando Qwen2.5-VL-7B...')
qwen_processor = AutoProcessor.from_pretrained(QWEN_ID)
qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    QWEN_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
qwen_model.eval()
print(f'Qwen cargado — VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# Evaluar sobre todos los estudios
metrics_qwen = run_zeroshot(
    model_name = 'Qwen2.5-VL-7B',
    generate_fn = generate_qwen,
    output_key = 'o1_qwen_7b',
    eval_df = meta_df,
    skip_if_exists= True,
)

# Liberar VRAM
del qwen_model, qwen_processor
gc.collect()
torch.cuda.empty_cache()
vram_free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
print(f'VRAM disponible: {vram_free:.1f} GB')

In [ ]:
# ── Modelo 3: HuatuoGPT-Vision 7B ────────────────────────────────────────────
# Arquitectura : idéntica a Qwen2.5-VL (misma clase HuggingFace)
# Preentrenamiento médico: PubMedVision 1.3M pares imagen-texto de PubMed

HUATUO_ID = 'FreedomIntelligence/HuatuoGPT-Vision-7B-Qwen2.5VL'

def generate_huatuo(img: Image.Image) -> str:
    """
    Genera un reporte mammográfico con HuatuoGPT-Vision 7B.
    Comparte arquitectura con Qwen2.5-VL — misma función de generación.
    """
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'text',  'text': SYSTEM_PROMPT},
            {'type': 'image', 'image': img},
            {'type': 'text',  'text': USER_PROMPT},
        ]
    }]
    text_in = huatuo_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, _ = process_vision_info(messages)
    inputs = huatuo_processor(
        text=[text_in], images=image_inputs,
        return_tensors='pt', padding=True,
    ).to(DEVICE)

    with torch.inference_mode():
        out = huatuo_model.generate(**inputs, max_new_tokens=300, do_sample=False)

    trimmed = out[0][inputs.input_ids.shape[1]:]
    return huatuo_processor.decode(trimmed, skip_special_tokens=True).strip()


# Cargar HuatuoGPT-Vision (usa la misma clase que Qwen2.5-VL)
print('Cargando HuatuoGPT-Vision-7B...')
huatuo_processor = AutoProcessor.from_pretrained(HUATUO_ID)
huatuo_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    HUATUO_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
huatuo_model.eval()
print(f'HuatuoGPT cargado — VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# Evaluar sobre todos los estudios
metrics_huatuo = run_zeroshot(
    model_name = 'HuatuoGPT-Vision-7B',
    generate_fn = generate_huatuo,
    output_key = 'o1_huatuo_7b',
    eval_df = meta_df,
    skip_if_exists= True,
)

# Liberar VRAM
del huatuo_model, huatuo_processor
gc.collect()
torch.cuda.empty_cache()
vram_free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
print(f'VRAM disponible: {vram_free:.1f} GB')

In [ ]:
# ── Tabla comparativa — resultados zero-shot ──────────────────────────────────
# Consolida las métricas de los 3 modelos en una tabla.

o1_results = {}
for key, name in [
    ('o1_medgemma_4b', 'MedGemma-4B-it'),
    ('o1_qwen_7b', 'Qwen2.5-VL-7B'),
    ('o1_huatuo_7b', 'HuatuoGPT-Vision-7B'),
]:
    path = f'{RESULTS_DIR}/{key}.json'
    if Path(path).exists():
        with open(path) as f:
            o1_results[name] = json.load(f)['metrics']
    else:
        print(f'{key}.json no encontrado — ejecuta la celda del modelo primero')

if o1_results:
    rows = []
    for name, r in o1_results.items():
        rows.append({
            'Modelo': name,
            'BERTScore F1': f'{r["bertscore_f1"]:.4f}',
            'CI 95%': f'[{r["bertscore_ci_low"]:.4f}–{r["bertscore_ci_high"]:.4f}]',
            'ROUGE-L': f'{r["rouge_l"]:.4f}',
            'BI-RADS Acc': f'{r["birads_acc"]:.4f}',
            'BI-RADS Macro F1': f'{r["birads_macro_f1"]:.4f}',
            'Density Acc': f'{r["density_acc"]:.4f}',
            'Density Macro F1': f'{r["density_macro_f1"]:.4f}',
            'N evaluados': r['n_evaluated'],
        })

    tabla = pd.DataFrame(rows)
    print('\n' + '='*100)
    print('  TABLA — COMPARATIVA ZERO-SHOT (VinDr-Mammo)')
    print('='*100)
    print(tabla.to_string(index=False))

    # Guardar como CSV para incluir en la tesis
    out_tabla = f'{RESULTS_DIR}/tabla_o1_comparativa.csv'
    tabla.to_csv(out_tabla, index=False)
    print(f'\nTabla guardada en: {out_tabla}')
else:
    print('No se encontraron resultados. Ejecuta las celdas de evaluación primero.')

In [ ]:
# ── Análisis de distribución de predicciones BI-RADS por modelo ───────────────
# Detecta sesgo de predicción: ¿algún modelo predice siempre la misma clase?

print('Distribución de predicciones BI-RADS por modelo:')
print('=' * 60)

for key, name in [
    ('o1_medgemma_4b', 'MedGemma-4B-it'),
    ('o1_qwen_7b', 'Qwen2.5-VL-7B'),
    ('o1_huatuo_7b', 'HuatuoGPT-Vision-7B'),
]:
    path = f'{RESULTS_DIR}/{key}.json'
    if not Path(path).exists():
        print(f' {name}: archivo no encontrado')
        continue

    with open(path) as f:
        data = json.load(f)

    # Extraer predicciones parseadas
    preds = [
        norm_birads(r.get('parsed', {}).get('birads'))
        for r in data['raw_results']
    ]
    preds = [p for p in preds if p]
    dist = Counter(preds)
    total = sum(dist.values())
    n_valid = data['metrics']['n_birads_valid']

    print(f'\n{name} (n_valid={n_valid}):')
    for birads in ['1', '2', '3', '4', '5']:
        n   = dist.get(birads, 0)
        pct = n / total * 100 if total > 0 else 0
        bar = '█' * int(pct / 2)
        print(f' BI-RADS {birads}: {n:4d} ({pct:5.1f}%) {bar}')

    # Detectar colapso: >60% en una sola clase
    dom_class, dom_cnt = dist.most_common(1)[0]
    dom_pct = dom_cnt / total * 100 if total > 0 else 0
    if dom_pct > 60:
        print(f' SESGO DETECTADO: BI-RADS {dom_class} domina con {dom_pct:.1f}%')

In [ ]:
# ── Ejemplos cualitativos de generación ───────────────────────────────────────
# Muestra 3 outputs crudos por modelo para análisis cualitativo.

N_EXAMPLES = 3
for key, name in [
    ('o1_medgemma_4b', 'MedGemma-4B-it'),
    ('o1_qwen_7b', 'Qwen2.5-VL-7B'),
    ('o1_huatuo_7b', 'HuatuoGPT-Vision-7B'),
]:
    path = f'{RESULTS_DIR}/{key}.json'
    if not Path(path).exists():
        continue

    with open(path) as f:
        data = json.load(f)

    print(f'\n{"="*70}')
    print(f'  {name} — ejemplos cualitativos')
    print(f'{"="*70}')

    for i, r in enumerate(data['raw_results'][:N_EXAMPLES]):
        sid = r['study_id']
        gt_row = meta_df[meta_df['study_id'] == sid]
        gt_b = gt_row['birads_norm'].iloc[0]  if not gt_row.empty else '?'
        gt_d = gt_row['density_norm'].iloc[0] if not gt_row.empty else '?'
        parsed = r.get('parsed', {})
        pred_b = norm_birads(parsed.get('birads'))  or 'UNKNOWN'
        pred_d = norm_density(parsed.get('breast_density')) or 'UNKNOWN'

        print(f'\n  Caso {i+1}: {sid}')
        print(f'  GT   → BI-RADS: {gt_b} | Density: {gt_d}')
        print(f'  PRED → BI-RADS: {pred_b} | Density: {pred_d}')
        print(f'  Output raw (primeros 400 chars):')
        print(f'  {r["raw_output"][:400]}')